# End-to-End Pipeline

Runs a single article through the full NLP pipeline in the correct call order:

```
raw article
    │
    ▼
 /extract  ──────────────────────────────► embedding_raw
    │                                           │
    │ extract                                   ▼
    ▼                                   /dedup-check-embed
 /summarize  ──► headline + summary
    │
    │ summary
    ├──────────────────────────────────► /classify  ──► topics, out_of_scope, geo_scope
    │
    │ text
    └──────────────────────────────────► /geotag    ──► geo_cities, geo_streets
```

**Prerequisite:** NLP service + Ollama running. `/readyz` → 200.

In [9]:
import sys, time, json, requests
sys.path.insert(0, '.')
from _scorecard import NLP_BASE_URL, HEADERS

# ── Article under test ──────────────────────────────────────────────────────
ARTICLE = {
    'article_id':   'e2e-001',
    'text': (
        'El Ayuntamiento de Madrid ha aprobado esta semana la ampliación del carril bici '
        'en la Gran Vía, una de las arterias principales de la capital. La nueva infraestructura '
        'ciclista se extenderá desde la plaza de España hasta la calle Alcalá, con una longitud '
        'total de 1,3 kilómetros. Las obras, que comenzarán el próximo mes de junio, tienen un '
        'presupuesto de 2,4 millones de euros y se prolongarán durante aproximadamente tres '
        'semanas. El concejal de Movilidad Sostenible ha destacado que este proyecto forma parte '
        'del Plan de Movilidad Urbana 2025-2030, que prevé la construcción de 150 kilómetros '
        'adicionales de carril bici en los próximos cinco años.'
    ),
    'raw_headline': 'Madrid amplía el carril bici en Gran Vía',
    'source':       'ayuntamiento_madrid',
}
print(f"Article: {ARTICLE['article_id']} — {ARTICLE['raw_headline']}")

Article: e2e-001 — Madrid amplía el carril bici en Gran Vía


In [10]:
r = requests.get(f'{NLP_BASE_URL}/readyz', headers=HEADERS)
assert r.status_code == 200, f'Service not ready: {r.status_code} {r.text}'
print('Service ready')

Service ready


In [12]:
# ── Step 1: /extract ─────────────────────────────────────────────────────────
t0 = time.monotonic()
resp = requests.post(
    f'{NLP_BASE_URL}/extract',
    json={'article_id': ARTICLE['article_id'], 'text': ARTICLE['text']},
    headers=HEADERS,
)
print(f'/extract  {resp.status_code}  {time.monotonic()-t0:.2f}s')
assert resp.status_code == 200, resp.text
extract_result = resp.json()

extract      = extract_result['extract']
embedding_raw = extract_result['embedding_raw']

print(f'  extract ({len(extract)} chars): {extract[:120]}...')
print(f'  embedding_raw: {len(embedding_raw)}-dim vector')

/extract  500  0.11s


AssertionError: Internal Server Error

In [13]:
# ── Step 2: /dedup-check-embed ───────────────────────────────────────────────
t0 = time.monotonic()
resp = requests.post(
    f'{NLP_BASE_URL}/dedup-check-embed',
    json={'article_id': ARTICLE['article_id'], 'embedding_raw': embedding_raw},
    headers=HEADERS,
)
print(f'/dedup-check-embed  {resp.status_code}  {time.monotonic()-t0:.2f}s')
assert resp.status_code == 200, resp.text
dedup_result = resp.json()

print(f'  duplicate_of={dedup_result["duplicate_of"]!r}  '
      f'stage={dedup_result["stage"]!r}  '
      f'score={dedup_result["score"]}  '
      f'indexed={dedup_result["indexed"]}')

/dedup-check-embed  200  0.11s
  duplicate_of='sum-001'  stage='embedding'  score=0.9671825170516968  indexed=False


In [14]:
# ── Step 3: /geotag ──────────────────────────────────────────────────────────
t0 = time.monotonic()
resp = requests.post(
    f'{NLP_BASE_URL}/geotag',
    json={
        'article_id': ARTICLE['article_id'],
        'text':       ARTICLE['text'],
        'headline':   ARTICLE['raw_headline'],
        'source':     ARTICLE['source'],
    },
    headers=HEADERS,
)
print(f'/geotag  {resp.status_code}  {time.monotonic()-t0:.2f}s')
assert resp.status_code == 200, resp.text
geo_result = resp.json()

print(f'  geo_scope={geo_result["geo_scope"]!r}')
print(f'  geo_cities={[c["city_name"] for c in geo_result["geo_cities"]]}')
print(f'  geo_streets={[s["span"] for s in geo_result["geo_streets"]]}')
print(f'  all_places={[p["text"] for p in geo_result["all_places"]]}')

/geotag  200  0.51s
  geo_scope='city'
  geo_cities=['Madrid']
  geo_streets=['plaza de España', 'calle Alcalá']
  all_places=['Madrid', 'Gran Vía', 'Gran Vía', 'plaza de España', 'calle Alcalá']


In [15]:
# ── Step 4: /summarize ───────────────────────────────────────────────────────
t0 = time.monotonic()
resp = requests.post(
    f'{NLP_BASE_URL}/summarize',
    json={
        'article_id': ARTICLE['article_id'],
        'text':       ARTICLE['text'],
        'extract':    extract,
        'headline':   ARTICLE['raw_headline'],
    },
    headers=HEADERS,
)
print(f'/summarize  {resp.status_code}  {time.monotonic()-t0:.1f}s')
assert resp.status_code == 200, resp.text
sum_result = resp.json()

print(f'  headline : {sum_result["headline"]}')
print(f'  summary  : {sum_result["summary"]}')

/summarize  200  44.3s
  headline : Madrid amplía el carril bici en la Gran Vía
  summary  : El Ayuntamiento de Madrid aprobó la ampliación del carril bici en la Gran Vía, que se extenderá desde la plaza de España hasta la calle Alcalá. Las obras, con un presupuesto de 2,4 millones de euros, comenzarán en junio y durarán aproximadamente tres semanas. Este proyecto forma parte del Plan de Movilidad Urbana 2025-2030.


In [16]:
# ── Step 5: /classify ────────────────────────────────────────────────────────
t0 = time.monotonic()
resp = requests.post(
    f'{NLP_BASE_URL}/classify',
    json={
        'article_id': ARTICLE['article_id'],
        'summary':    sum_result['summary'],
        'geo_cities': geo_result['geo_cities'],
        'search_tags': [],
        'source_profile': None,
        'geo_scope':  geo_result['geo_scope'],
    },
    headers=HEADERS,
)
print(f'/classify  {resp.status_code}  {time.monotonic()-t0:.2f}s')
assert resp.status_code == 200, resp.text
cls_result = resp.json()

print(f'  topics={cls_result["topics"]}')
print(f'  geo_scope={cls_result["geo_scope"]!r}  (from geotagger)')
print(f'  out_of_scope={cls_result["out_of_scope"]}')

/classify  200  0.67s
  topics=['movilidad sostenible', 'infraestructura ciclista', 'obras de movilidad']
  geo_scope='city'  (from geotagger)
  out_of_scope=False


In [17]:
# ── Pipeline summary ─────────────────────────────────────────────────────────
print('=' * 60)
print('  PIPELINE RESULT')
print('=' * 60)
print(f'  article_id   : {ARTICLE["article_id"]}')
print(f'  headline     : {sum_result["headline"]}')
print(f'  summary      : {sum_result["summary"]}')
print(f'  topics       : {cls_result["topics"]}')
print(f'  geo_scope    : {cls_result["geo_scope"]}')
print(f'  out_of_scope : {cls_result["out_of_scope"]}')
print(f'  cities       : {[c["city_name"] for c in geo_result["geo_cities"]]}')
print(f'  streets      : {[s["span"] for s in geo_result["geo_streets"]]}')
print(f'  is_duplicate : {dedup_result["duplicate_of"] is not None}')
if dedup_result['duplicate_of']:
    print(f'  duplicate_of : {dedup_result["duplicate_of"]}  (stage={dedup_result["stage"]})')
print('=' * 60)

  PIPELINE RESULT
  article_id   : e2e-001
  headline     : Madrid amplía el carril bici en la Gran Vía
  summary      : El Ayuntamiento de Madrid aprobó la ampliación del carril bici en la Gran Vía, que se extenderá desde la plaza de España hasta la calle Alcalá. Las obras, con un presupuesto de 2,4 millones de euros, comenzarán en junio y durarán aproximadamente tres semanas. Este proyecto forma parte del Plan de Movilidad Urbana 2025-2030.
  topics       : ['movilidad sostenible', 'infraestructura ciclista', 'obras de movilidad']
  geo_scope    : city
  out_of_scope : False
  cities       : ['Madrid']
  streets      : ['plaza de España', 'calle Alcalá']
  is_duplicate : True
  duplicate_of : sum-001  (stage=embedding)
